# What is fuzzy string matching?

It is a method to match two strings that are not exactly the same but are similar. The aim could be to fix spelling mistakes, or finding two words that mean the same written differently. It can be done in multiple ways one of which is called the edit distance, and another is N-gram matching, also there is token based matching. 


# What is a distance measure for strings?

A distance measure is a way to quantify how far two strings are from one another, there more than one way of doing this, the simplest example is the hamming distance, there is also the edit distance.

Edit distance (also known as Levenshtein distance) matching is a method that measures the distance from one string to another by turning one into the other, by using a combination of operations deleting, inserting and replacing (which is the two instering after deleting). Insertion and deletion cost 1 each and replacing costs 2 and so it finds the shortest path from one string to the other using these operations.

The hamming distance counts the differences between characters of two equal length strings, it is not the same as the edit distance from what I have seen.

In [2]:
def hamming_distance(first_string, second_string):
    """Return the number of positions containing different characters.

    Hamming distance is a restricted edit distance: it allows substitutions
    only, so the two strings must have the same length.
    """
    if not isinstance(first_string, str) or not isinstance(second_string, str):
        raise TypeError("Hamming distance expects two strings.")

    if len(first_string) != len(second_string):
        raise ValueError("Hamming distance requires strings of equal length.")

    return sum(
        first_character != second_character
        for first_character, second_character in zip(first_string, second_string)
    )


test_pairs = [
    ("karolin", "kathrin", 3),
    ("karolin", "kerstin", 3),
    ("1011101", "1001001", 2),
    ("tone", "tune", 1),
    ("same", "same", 0),
    ("", "", 0),
]

for first_string, second_string, expected_distance in test_pairs:
    calculated_distance = hamming_distance(first_string, second_string)
    assert calculated_distance == expected_distance
    print(
        f"{first_string!r} and {second_string!r}: "
        f"Hamming distance = {calculated_distance}"
    )

# Hamming distance is undefined for unequal-length strings because insertion
# and deletion are not allowed. Confirm that the function rejects this case.
try:
    hamming_distance("short", "longer")
except ValueError as error:
    print(f"Unequal-length test: {error}")
else:
    raise AssertionError("Unequal-length strings should raise ValueError.")

'karolin' and 'kathrin': Hamming distance = 3
'karolin' and 'kerstin': Hamming distance = 3
'1011101' and '1001001': Hamming distance = 2
'tone' and 'tune': Hamming distance = 1
'same' and 'same': Hamming distance = 0
'' and '': Hamming distance = 0
Unequal-length test: Hamming distance requires strings of equal length.


In [ ]:
# The hamming distance is not the same as the Levenshtein distance, 
# which allows insertions and deletions.The hamming distance measures how many 
# characters differ between two strings of equal length, while the Levenshtein distance measures the minimum number 
# of single-character edits (insertions, deletions, or substitutions) required to change one string into the other.

# What is N-gram distance?

N-gram matching is a method that turns strings into n character groups and sees how many n character groups match from both strings meaning the more n charatcer groups match out of the total unique n-grams groups this gives a similarity score and then user can decide wether or not they are the same string if they meet a threshold. But to get the distance measure between them you subtract the similarity calculated from 1 that gives the distance between them.

In [3]:
def create_character_ngrams(text, n):
    """Return the set of contiguous character N-grams in `text`."""
    if not isinstance(text, str):
        raise TypeError("N-gram creation expects a string.")
    if not isinstance(n, int) or isinstance(n, bool) or n <= 0:
        raise ValueError("n must be a positive integer.")

    return {text[index : index + n] for index in range(len(text) - n + 1)}


def ngram_distance(first_string, second_string, n=2):
    """Return the Jaccard distance between two character N-gram sets.

    A result of 0 means the N-gram sets are identical, while 1 means they
    have no N-grams in common.
    """
    first_ngrams = create_character_ngrams(first_string, n)
    second_ngrams = create_character_ngrams(second_string, n)

    all_ngrams = first_ngrams | second_ngrams
    if not all_ngrams:
        return 0.0 if first_string == second_string else 1.0

    shared_ngrams = first_ngrams & second_ngrams
    ngram_similarity = len(shared_ngrams) / len(all_ngrams)
    return 1.0 - ngram_similarity


ngram_test_pairs = [
    ("night", "nacht", 2),
    ("context", "contact", 2),
    ("similarity", "similar", 3),
    ("python", "python", 2),
    ("abc", "xyz", 2),
]

for first_string, second_string, n in ngram_test_pairs:
    first_ngrams = create_character_ngrams(first_string, n)
    second_ngrams = create_character_ngrams(second_string, n)
    distance = ngram_distance(first_string, second_string, n)

    assert 0.0 <= distance <= 1.0
    print("\n" + "-" * 70)
    print(f"Strings: {first_string!r} and {second_string!r}; n = {n}")
    print(f"First N-grams:  {sorted(first_ngrams)}")
    print(f"Second N-grams: {sorted(second_ngrams)}")
    print(f"Shared N-grams: {sorted(first_ngrams & second_ngrams)}")
    print(f"N-gram distance: {distance:.4f}")

assert ngram_distance("python", "python", 2) == 0.0
assert ngram_distance("abc", "xyz", 2) == 1.0


----------------------------------------------------------------------
Strings: 'night' and 'nacht'; n = 2
First N-grams:  ['gh', 'ht', 'ig', 'ni']
Second N-grams: ['ac', 'ch', 'ht', 'na']
Shared N-grams: ['ht']
N-gram distance: 0.8571

----------------------------------------------------------------------
Strings: 'context' and 'contact'; n = 2
First N-grams:  ['co', 'ex', 'nt', 'on', 'te', 'xt']
Second N-grams: ['ac', 'co', 'ct', 'nt', 'on', 'ta']
Shared N-grams: ['co', 'nt', 'on']
N-gram distance: 0.6667

----------------------------------------------------------------------
Strings: 'similarity' and 'similar'; n = 3
First N-grams:  ['ari', 'ila', 'imi', 'ity', 'lar', 'mil', 'rit', 'sim']
Second N-grams: ['ila', 'imi', 'lar', 'mil', 'sim']
Shared N-grams: ['ila', 'imi', 'lar', 'mil', 'sim']
N-gram distance: 0.3750

----------------------------------------------------------------------
Strings: 'python' and 'python'; n = 2
First N-grams:  ['ho', 'on', 'py', 'th', 'yt']
Second N-gram